# Experiment 8 - Detecting Three Straight Lines with the Hough Transform

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
H, W = 300, 400
canvas = np.zeros((H, W), dtype=np.uint8)
cv2.line(canvas, (20, 260), (380, 40), 255, 2)
cv2.line(canvas, (60, 30), (90, 280), 255, 2)
cv2.line(canvas, (10, 150), (390, 190), 255, 2)

rng = np.random.default_rng(11)
noise = rng.random(canvas.shape) < 0.01
scene = canvas.copy()
scene[noise] = 255

edges = cv2.Canny(scene, 50, 150)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(scene, cmap="gray")
axes[0].set_title("three lines + noise")
axes[1].imshow(edges, cmap="gray")
axes[1].set_title("canny edges")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
def hough(edge_img, theta_step=1.0):
    thetas = np.deg2rad(np.arange(-90, 90, theta_step))
    diag = int(np.ceil(np.hypot(*edge_img.shape)))
    rhos = np.arange(-diag, diag + 1)
    acc = np.zeros((len(rhos), len(thetas)), dtype=np.int32)
    ys, xs = np.nonzero(edge_img)
    cos_t, sin_t = np.cos(thetas), np.sin(thetas)
    r = np.round(np.outer(xs, cos_t) + np.outer(ys, sin_t)).astype(int) + diag
    cols = np.broadcast_to(np.arange(len(thetas)), r.shape)
    np.add.at(acc, (r.ravel(), cols.ravel()), 1)
    return acc, rhos, thetas

acc, rhos, thetas = hough(edges)

plt.figure(figsize=(8, 6))
plt.imshow(acc, cmap="inferno", aspect="auto",
           extent=[np.rad2deg(thetas[0]), np.rad2deg(thetas[-1]), rhos[-1], rhos[0]])
plt.xlabel("theta (deg)")
plt.ylabel("rho (px)")
plt.title("hough accumulator")
plt.colorbar()
plt.show()

In [ ]:
def top_peaks(acc, n=3, nbhd=(25, 15)):
    a = acc.copy()
    found = []
    for _ in range(n):
        r, t = np.unravel_index(np.argmax(a), a.shape)
        found.append((r, t, a[r, t]))
        a[max(0, r - nbhd[0]):r + nbhd[0] + 1, max(0, t - nbhd[1]):t + nbhd[1] + 1] = 0
    return found

def draw(img, lines, color):
    out = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    for rho, th in lines:
        c, s = np.cos(th), np.sin(th)
        x0, y0 = c * rho, s * rho
        p1 = (int(x0 - 1000 * s), int(y0 + 1000 * c))
        p2 = (int(x0 + 1000 * s), int(y0 - 1000 * c))
        cv2.line(out, p1, p2, color, 2)
    return out

peaks = top_peaks(acc, 3)
mine = [(rhos[r], thetas[t]) for r, t, _ in peaks]
for (rho, th), (_, _, votes) in zip(mine, peaks):
    print(f"rho={rho:5d}  theta={np.rad2deg(th):6.1f} deg  votes={votes}")

In [ ]:
cv_lines = cv2.HoughLines(edges, 1, np.pi / 180, 80)
cv_top = []
for rho, th in cv_lines.reshape(len(cv_lines), -1)[:, :2]:
    if all(abs(rho - r) > 20 or abs(th - t) > np.deg2rad(10) for r, t in cv_top):
        cv_top.append((rho, th))
    if len(cv_top) == 3:
        break

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, lines, t in zip(axes, [mine, cv_top], ["own hough - top 3", "cv2.HoughLines - top 3"]):
    ax.imshow(cv2.cvtColor(draw(scene, lines, (0, 0, 255)), cv2.COLOR_BGR2RGB))
    ax.set_title(t)
    ax.axis("off")
plt.show()

In [ ]:
segs = cv2.HoughLinesP(edges, 1, np.pi / 180, 60, minLineLength=80, maxLineGap=10)
out = cv2.cvtColor(scene, cv2.COLOR_GRAY2BGR)
for x1, y1, x2, y2 in segs.reshape(-1, 4):
    cv2.line(out, (x1, y1), (x2, y2), (0, 255, 0), 2)
print("segments found:", len(segs))

plt.figure(figsize=(6, 5))
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
plt.title("probabilistic hough segments")
plt.axis("off")
plt.show()